# Calculating a Typical Meteorological Year -- Methodology Walkthrough
<br>This notebook walks through the process of calculating a [Typical Meteorological Year](https://nsrdb.nrel.gov/data-sets/tmy), a specific kind of `Climate Profile` hourly dataset used for applications in energy and building systems modeling. Because this represents average rather than extreme conditions, an TMY dataset is not suited for designing systems to meet the worst-case conditions occurring at a location. 

The TMY methodology here mirrors that of the Sandia/NREL TMY3 methodology, and uses historic and projected downscaled climate data available through the Cal-Adapt: Analytics Engine catalog. As this methodology heavily weights the solar radiation input data, be aware that the final selection of "typical" months may not be typical for other variables. 

**Intended Application:** As a user, I want to <span style="color:#FF0000">**generate a typical meteorological year file**</span> for a location of interest:
- Understand the methods that are involved in generating a TMY dataset
- Visualize the TMY dataset across all input variables
- Export the TMY dataset for available models for input into my workflow

**Note**: 
1. This notebook is a **full demonstration** of the Typical Meteorological Year methodology, for full transparency.
2. For practical generation of a TMY dataset, please use the notebook `custom_climate_profiles.ipynb`, where a user can generate a custom climate profile by providing only a **location** and **reference period**. 

### Step 0: Set-up

Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [ ]:
import pandas as pd
import xarray as xr
import numpy as np
import hvplot.xarray
from importlib.resources import files
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt

import climakitae as ck
from climakitae.core.constants import UNSET
from climakitae.core.data_export import write_tmy_file

# Initialize ClimateData object
cd = ck.ClimateData(verbosity=-1)

### Step 1: Grab and process all required input data

The [TMY3 method](https://docs.nlr.gov/docs/fy08osti/43156.pdf) selects a "typical" month based on ten daily variables: max, min, and mean air and dew point temperatures, max and mean wind speed, global irradiance and direct irradiance.  

#### Step 1a: Select location of interest
TMYs are calculated for a specific location of interest, like a building or power plant. Here, we will use a known weather station location, via their latitude and longitude to extract the data that we need to calculate the TMY. In the example below, we will look specifically at Los Angeles International Airport, but will note in the code below how you can provide your own location coordinates too. 

In [ ]:
# read in station file of CA HadISD stations
stn_filepath_s3 = "s3://cadcat/hadisd/hadisd_stations.csv"
stn_file = pd.read_csv(stn_filepath_s3, index_col=[0])
stn_file.head(5)

In [ ]:
# grab airport
stn_name = "Los Angeles International Airport (KLAX)"
stn_code = stn_file.loc[stn_file["station"] == stn_name]["station id"].item()
one_station = stn_file.loc[stn_file["station"] == stn_name]
stn_lat = one_station.LAT_Y.item()
stn_lon = one_station.LON_X.item()
stn_state = one_station.state.item()

# Output results
print("station coordinates:", (stn_lat, stn_lon))
print("station code:", stn_code)

Alternatively, you may want to provide your own location instead of one of the HadISD stations above. If so, uncomment the cell below by removing the `#` symbol and modifying the lines of code. Note, with custom locations, if an elevation is not provided, a default value of 0.0 m will be used. 

In [ ]:
## provide your own location, via latitude and longitude coordinates
# stn_lat = YOUR_LAT_HERE
# stn_lon = YOUR_LON_HERE
# stn_state = 'YOUR_STATE_HERE'
# stn_name = 'YOUR_STATION_NAME_HERE'
# stn_code = 'custom'
# stn_elev = YOUR_ELEV_HERE # meters

#### Step 1b: Select time frame of interest
The second required input for generating a TMY dataset is the **time frame of interest**. The recommended minimum number of input years for a TMY dataset is 15-20 years worth of daily data. For data post-2014, we will utilize SSP 3-7.0, although scenario selection in the near-future is relatively independent. If calculating a TMY for the far-future, **carefully consider which scenario SSP to include**, as there will be **significant** differences present. 

As an example, we will process the data for our designated station location (latitude, and longitude) at 3 km over the <span style="color:#FF0000">2005-2020</span> period in a time-based approach. 

> **Note:** We use a 15 year period here to keep the notebook runtime manageable, as this is a methods demonstration only. For production use, a 30-year period is recommended to represent a standard climatological baseline.

In [ ]:
# selected reference period
start_year = 2005
end_year = 2020

#### Step 1c: Retrieve variables in catalog
It is important to note that not all models in the Cal-Adapt: Analytics Engine have the solar variables critical for TMY file generation - in fact, only 4 do! We will carefully subset our variables to ensure that the same 4 models are selected for consistency. 

Lastly, because the dynamically downscaled WRF data in the Cal-Adapt: Analytics Engine is in UTC time, we'll convert to the timezone of the station we've selected. This is particularly important for the timing of solar radiation max and min, which is a critical piece of a TMY. The handy `convert_to_local_time` processor in the `ClimateData` object corrects for this, and ensures that all data are within the same daily timestamp.

In [ ]:
# selected models
data_models = [
    "wrf_ucla_taiesm1_ssp370_r1i1p1f1",
    "wrf_ucla_ec-earth3_ssp370_r1i1p1f1",
    "wrf_ucla_miroc6_ssp370_r1i1p1f1",
    "wrf_ucla_mpi-esm1-2-hr_ssp370_r3i1p1f1",
]

# map variable names to descriptive names and units
var_info = {
    "t2": {"long_name": "Air temperature at 2m", "units": "degC"},
}

Now that we have set up the model list, let's start retrieving data! We will need to calculate summary statistics and reduce to daily timescales for the following variables: 
- Air temperature (`t2`)
- Wind speed (`wind_speed_10m`)
- Dew point temperature (`dew_point_2m`)
- Instantaneous downwelling shortwave flux at Earth's surface (`swdnb`)
- Shortwave surface downward direct normal irradiance (`swddni`)

In [ ]:
t2_ds = (
    cd.catalog("cadcat")
    .variable("t2")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "convert_units": "degC",
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

In [ ]:
# Subset simulations and convert to DataArray
t2_ds = t2_ds.sel(sim=data_models, time=slice(str(start_year), str(end_year))).t2

# Load into memory using dask progress bar
with ProgressBar():
    t2_ds.load()

In [ ]:
t2_ds.to_netcdf("hourly_air_temp_lax_2005_2020.nc")

#### Step 1d: Remoev Pinatubo Years

In [ ]:
t2_ds = xr.load_dataset("hourly_air_temp_lax_2005_2020.nc")

In [ ]:
air_temp = t2_ds.where(~t2_ds['time'].dt.year.isin([1991, 1992, 1993, 1994]))

### Step 3: Take Percentile

In [138]:
def compute_profile(data: xr.DataArray, days_in_year: int = 365, q=0.5) -> pd.DataFrame:
    """
    Calculates the standard year climate profile for warming level data using 8760
    analysis.

    This function handles global warming levels approach using time_delta coordinate.
    Processes all 30 years of warming level data centered around the year a warming level
    is reached, computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns a characteristic profile of 8760 hours (one year) for each warming level
    and simulation combination.

    Parameters
    ----------
    data : xr.DataArray
        Hourly base-line subtracted data for one variable with warming_level,
        time_delta, and simulation dimensions. Expected to contain ~30 years
        (262,800 hours) of data for each warming level and simulation.

    days_in_year : int, optional
        Either 366 or 365, depending on whether or not the year is a leap year.
        Default to 365 days

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).
        Default is 0.5 (median).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Select air temperature as the variable used to generate candidate years
    data = data['t2']

    # Check for simulation dimension
    has_simulation = "sim" in data.dims
    if has_simulation:
        simulations = data.sim.values
    else:
        simulations = [None]

    # Get all available time data (all 30 years)
    hours_per_day = 24
    hours_per_year = 8760
    total_hours = len(data.time)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time", hour_of_year_all))

    # Initialize storage for profiles
    store_closest_year_idx = {}
    store_sel_year = {}
    df_list = []
    for sim_idx, sim in enumerate(simulations):

        # Select data for this warming level and simulation combination
        if has_simulation:
            subset_data = data.isel(sim=sim_idx)
        else:
            subset_data = data
            
        # Vectorized quantile computation using numpy
        # Reshape raw values into (n_years, hours_per_year) then compute
        # the quantile across years for each hour-of-year position
        values = subset_data.values
        n_total = len(values)
        usable = (n_total // hours_per_year) * hours_per_year
        year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

        # Compute quantile targets for each of the 8760 hour positions
        quantile_targets = np.nanquantile(
            year_hour_matrix, q, axis=0
        )  # shape: (8760,)

        # For each hour position, find the actual year whose value is
        # closest to the quantile (avoids interpolation)
        diffs = np.abs(
            year_hour_matrix - quantile_targets[np.newaxis, :]
        )  # (n_years, 8760)

        closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)

        # Calendar year for each hour-of-year position
        usable_times = subset_data.time.values[:usable]
        row_years = pd.DatetimeIndex(usable_times).year.values.reshape(-1, hours_per_year)[:, 0]
        sel_year = row_years[closest_year_idx]  # (8760,) — calendar year, not row index

        # Store the selected years in a pandas dataframe
        df_i = pd.DataFrame({
            "hour": np.arange(1, hours_per_year + 1),
            "sim": sim,
            "year": sel_year.astype(int),
        })
        df_list.append(df_i)

    # Concatenate list together for all simulations
    top_df = pd.concat(df_list).reset_index(drop=True)

    return top_df


In [150]:
top_df = compute_profile(data=air_temp,q=0.5)

      📊 Processing 140,160 hours (16 years) of data
      🎯 Computing 50th percentile for each hour of year


### Step 4: Generate the Persistence XMY data outputs

Generally, the following data is outputted using the TMY months:
- Date & time (UTC)
- Air temperature at 2m [°C]
- Dew point temperature [°C]
- Relative humidity [%]
- Global horizontal irradiance [W/m2]
- Direct normal irradiance [W/m2]
- Diffuse horizontal irradiance [W/m2]
- Downwelling infrared radiation [W/m2]
- Wind speed at 10m [m/s]
- Wind direction at 10m [°]
- Surface air pressure [Pa]

The following function will retrieve all of this data for the designated TMY month and concatenate it together into a pandas dataframe so that we can export it as a csv file. We'll do this next. 

In [ ]:
def generate_persistence_xmy_data(
    top_df: pd.DataFrame,
    start_year: int,
    end_year: int,
    lat: float,
    lon: float,
    data_models: list,
    hrs_in_a_day: int,
) -> dict:
    """Generate typical meteorological year data.

    Parameters
    ----------
    top_df : pd.DataFrame
        Table with columns month, simulation, and year. Each month-sim-yr
        combo represents the top candidate with the lowest weighted FS statistic.
    start_year : int
    end_year : int
    lat : float
    lon : float
    data_models : list
    hrs_in_a_day: int

    Returns
    -------
    dict of str : pd.DataFrame
        Dictionary in the format {simulation: TMY DataFrame}.
    """
    # Define climate data object with minimal terminal output
    cd = ck.ClimateData(verbosity=-2)

    # Define variables and unit conversions
    var_info = {
        "t2": {
            "long_name": "Air temperature at 2m (degC)",
            "units": "degC",
        },  # convert default K -> C
        "dew_point_2m": {
            "long_name": "Dew point temperature at 2m (degC)",
            "units": "degC",
        },  # convert default K -> C
        "relative_humidity_2m": {
            "long_name": "Relative humidity (0-100)",
            "units": UNSET,
        },  # Use unit default (0-100)
        "swdnb": {
            "long_name": "Instantaneous downwelling shortwave flux at bottom (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "swddni": {
            "long_name": "Shortwave surface downward direct normal irradiance (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "swddif": {
            "long_name": "Shortwave surface downward diffuse irradiance (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "lwdnb": {
            "long_name": "Instantaneous downwelling longwave flux at bottom (W/m2)",
            "units": UNSET,
        },  # Use unit default (W/m2)
        "wind_speed_10m": {
            "long_name": "Wind speed at 10m (m/s)",
            "units": UNSET,
        },  # Use unit default (m/s)
        "wind_direction_10m": {
            "long_name": "Wind direction at 10m (degrees)",
            "units": UNSET,
        },  # Use unit default (degrees)
        "psfc": {
            "long_name": "Surface pressure (Pa)",
            "units": UNSET,
        },  # Use unit default (Pa)
    }

    ## ================== GET DATA FROM CATALOG ==================
    print(
        f"STEP 1: RETRIEVING HOURLY DATA FROM CATALOG FOR {len(var_info)} VARIABLES\n"
    )
    all_vars_list = []
    for i, (var, info) in enumerate(var_info.items(), start=1):
        long_name = info["long_name"]
        units = info["units"]
        print(f"({i}/{len(var_info)}) Variable: {var}")
        print("Retrieving data from catalog...")
        data_by_var = (
            cd.catalog("cadcat")
            .variable(var)
            .activity_id("WRF")
            .institution_id("UCLA")
            .table_id("1hr")
            .grid_label("d03")
            .experiment_id(["historical", "ssp370"])
            .processes(
                {
                    "time_slice": (start_year, end_year + 1),
                    "clip": (lat, lon),
                    "convert_units": units,
                    "convert_to_local_time": {"convert": "yes"},
                }
            )
            .get()
        )

        # Subset simulations and convert to DataArray
        data_by_var = data_by_var.sel(
            sim=data_models, time=slice(str(start_year), str(end_year))
        )[var]

        # Update variable name to use long_name for clarity
        data_by_var.name = long_name

        print(f"Retrieved. Loading data into memory...")
        with ProgressBar():
            data_by_var.load()

        print(f"Data loaded into memory.\n")
        all_vars_list.append(data_by_var)

    # Merge data from all variables into a single xr.Dataset object
    # Drop unneeded coordinates
    all_vars_ds = xr.merge(all_vars_list)
    all_vars_ds = all_vars_ds.drop_vars(
        ["lakemask", "landmask", "x", "y", "Lambert_Conformal"], errors="ignore"
    )

    # Clear individual DataArrays from memory after merge
    del all_vars_list

    ## ================== CONSTRUCT TMY ==================
    print("\nSTEP 2: CALCULATING PERSISTENCE EXTREME METEOROLOGICAL YEAR PER MODEL SIMULATION\n")
    tmy_df_all = {}
    for sim in all_vars_ds.sim.values:
        df_list = []
        print(f"Calculating persistence XMY for simulation: {sim}")
        for hr in np.arange(1,hrs_in_a_day,1):
            # Get year corresponding to month and simulation combo
            year = top_df.loc[
                (top_df["hour"] == hr) & (top_df["sim"] == sim)
            ].year.item()

            # Select data for unique hour, month, year, and simulation
            data_at_stn_sim_yr = all_vars_ds.sel(
                sim=sim,
                time=(all_vars_ds.time.dt.year == year),
            ).expand_dims("sim")

            # Add "hour" coordinate, for hour of year
            data_at_stn_sim_yr_by_hr  = data_at_stn_sim_yr.assign_coords(hour=("time", np.arange(1, len(data_at_stn_sim_yr.time) + 1)))

            #! for testing, store year values in their own coordinate
            data_at_stn_sim_yr_by_hr  = data_at_stn_sim_yr_by_hr.assign_coords(selected_year=year)

            data_at_stn_hr_sim_yr = data_at_stn_sim_yr_by_hr.sel(time=(data_at_stn_sim_yr_by_hr.hour == hr))

            # Reformat as dataframe
            df_by_hr_sim_yr = data_at_stn_hr_sim_yr.to_dataframe().reset_index()

            # Reformat time index to remove seconds
            df_by_hr_sim_yr["time"] = pd.to_datetime(
                df_by_hr_sim_yr["time"].values
            ).strftime("%Y-%m-%d %H:%M")

            df_list.append(df_by_hr_sim_yr)

        # Concatenate all DataFrames together
        tmy_df_by_sim = pd.concat(df_list)
        tmy_df_all[sim] = tmy_df_by_sim

        print("TMY PROCESS COMPLETE.")

    return tmy_df_all

In the next cell, we will run the `generate_tmy_data` function which will retrieve, subset, and format the data for each month according to the TMY months for all requested variables. We have included print statements so you can watch the progress for each variable in each month as it builds. 

<span style="color:#FF0000">**Note**: <span style="color:#000000"> This will take time! On the Analytics Engine JupyterHub, this takes approximately 22 minutes. Progress bars will indicate the status of generating the TMY data for each simulation. 

In [ ]:
tmy_data_to_export = generate_persistence_xmy_data(
    top_df, start_year, end_year, stn_lat, stn_lon, data_models
)

Let's observe what the TMY data looks like for one of the simulations:

In [ ]:
simulation = "wrf_ucla_miroc6_ssp370_r1i1p1f1"
tmy_data_to_export[simulation].head(5)

Next, we visualize the TMY data itself.

In [ ]:
# Make plot using pandas
ax = tmy_data_to_export[simulation].plot(
    x="time",
    y=[
        col
        for col in tmy_data_to_export[simulation]
        if col not in ["time", "sim", "lat", "lon"]
    ],
    subplots=True,
    figsize=(12, 12),
    legend=True,
    fontsize=10,
)

# Plot formatting
for a in ax.flatten():
    a.xaxis.label.set_fontsize(12)
    a.yaxis.label.set_fontsize(12)
    a.legend(loc="upper right", fontsize=14)
plt.suptitle(
    f"Typical Meteorological Year (simulation: {simulation})", fontsize=16, y=0.98
)
plt.tight_layout();

Lastly, let's export the TMY data below as csv files. There will be a file per simulation downloaded. When utilizing TMY data in your own workflows, we recommend that **all simulations are considered** in your analyses, especially for future scenarios. Note, if you are working with a custom location, please also provide the optional `stn_elev` argument to `write_tmy_file`; if no elevation value is provided, an elevation value of 0.0 is set as the default. 

In [ ]:
for sim, tmy in tmy_data_to_export.items():
    filename = "TMY_{0}_{1}".format(
        stn_name.replace(" ", "_").replace("(", "").replace(")", ""), sim
    ).lower()
    write_tmy_file(
        filename,
        tmy_data_to_export[sim],
        (start_year, end_year),
        stn_name,
        stn_code,
        stn_lat,
        stn_lon,
        stn_state,
        file_ext="epw",
    )